# OpenSOS — Figure Generator

Re-runnable notebook that recreates **every figure** in the OpenSOS paper. Edit the data in the first code cell and *Run All* to regenerate.

**Install once:** `pip install matplotlib pillow numpy`

**Folders:** app screenshots live in `screens/`; outputs are written to `figures/`.

| Notebook output | Paper figure |
|---|---|
| `fig1_architecture.png` | Figure 1 — system architecture |
| `fig2_screens_core.png` | Figure 2 — app screenshots (core flow) |
| `fig3_screens_setup.png` | Figure 3 — app screenshots (setup) |
| `fig4_os_share.png` | Figure 4 — mobile OS market share |
| `fig5_coverage.png` | Figure 5 — feature availability by platform |
| `fig6_reach.png` | Figure 6 — cross-platform reach |

To refresh the screenshots themselves, run `npm run preview:screenshots` in the app repo and copy the new PNGs into `screens/`.


In [ ]:

# === OpenSOS figure generator — configuration (edit values here) ===
import os, numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from PIL import Image, ImageDraw, ImageFont

SCREENS_DIR = "screens"      # app screenshots (from `npm run preview:screenshots`)
FIG_DIR     = "figures"      # output folder
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({"font.family":"DejaVu Sans","font.size":11,"axes.edgecolor":"#444"})

# palette
RED, BLUE, GREEN, AMBER, GREY, DARK = "#dc2626","#2563eb","#15803d","#b45309","#9aa2ac","#1a1d21"

# ---- editable data ----
OS_SHARE = {                 # mobile OS market share (%)  [update from StatCounter]
    "Worldwide":     {"Android":70.4, "iOS":29.3},
    "United States": {"Android":40.0, "iOS":59.8},
}
# feature availability by platform: 2=Yes, 1=Partial, 0=No
COVERAGE_COLS = ["Android\n(Chrome)","iPhone / iPad\n(Safari)","Desktop\n(Chrome/Edge)"]
COVERAGE = [
    ("On-screen SOS (press & hold)", [2,2,2]),
    ("Wi-Fi relay trigger",          [2,2,2]),
    ("Email alert + GPS location",   [2,2,2]),
    ("Installable app (home screen)",[2,2,2]),
    ("Works offline (cached)",       [2,2,2]),
    ("Bluetooth button (optional)",  [2,0,2]),
]
REACH = [   # (label, percent, is_optional)
    ("Web app + email alert\n+ GPS (any phone)", 99.6, False),
    ("Installable PWA\n(home screen)",           99.0, False),
    ("Wi-Fi remote trigger\n(any browser)",      99.6, False),
    ("Bluetooth button\n(optional extra)",       76.0, True),
]

def montage(items, outname, target_h=760, cols=4):
    def font(sz):
        for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
                  "C:/Windows/Fonts/arialbd.ttf","/Library/Fonts/Arial Bold.ttf",
                  "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"]:
            if os.path.exists(p):
                try: return ImageFont.truetype(p, sz)
                except: pass
        return ImageFont.load_default()
    imgs=[]
    for fn,label in items:
        im=Image.open(os.path.join(SCREENS_DIR,fn)).convert("RGB")
        w,h=im.size; im=im.crop((0,0,w,min(h,int(w*1.9))))
        s=target_h/im.size[1]; im=im.resize((int(im.size[0]*s),target_h)); imgs.append((im,label))
    pad,cap,bd=26,42,2
    cw=max(i.size[0] for i,_ in imgs); rows=(len(imgs)+cols-1)//cols
    W=cols*cw+(cols+1)*pad; H=rows*(target_h+cap)+(rows+1)*pad
    canvas=Image.new("RGB",(W,H),"white"); d=ImageDraw.Draw(canvas); f=font(26)
    for idx,(im,label) in enumerate(imgs):
        r,c=divmod(idx,cols)
        x=pad+c*(cw+pad)+(cw-im.size[0])//2; y=pad+r*(target_h+cap+pad)
        canvas.paste(im,(x,y))
        d.rectangle([x-bd,y-bd,x+im.size[0]+bd,y+target_h+bd],outline="#d3d8de",width=bd)
        tw=d.textlength(label,font=f)
        d.text((pad+c*(cw+pad)+(cw-tw)//2, y+target_h+8), label, fill="#1a1d21", font=f)
    out=os.path.join(FIG_DIR,outname); canvas.save(out); print("wrote",out,canvas.size); return out

print("Config loaded. Output ->", FIG_DIR)

In [ ]:

# === Figure 1 — system architecture ===
fig,ax=plt.subplots(figsize=(9,4.6)); ax.axis("off"); ax.set_xlim(0,10); ax.set_ylim(0,6)
def box(x,y,w,h,t,fc,tc="white",fs=10):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=0.02,rounding_size=0.12",fc=fc,ec="none"))
    ax.text(x+w/2,y+h/2,t,ha="center",va="center",color=tc,fontsize=fs,weight="bold")
def arr(x1,y1,x2,y2): ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle="-|>",mutation_scale=14,color="#555",lw=1.6))
box(0.2,4.4,2.5,0.8,"Press & hold SOS",RED)
box(0.2,3.1,2.5,0.8,"Wi-Fi relay trigger",BLUE)
box(0.2,1.8,2.5,0.8,"Bluetooth button\n(optional, Android)",GREY,fs=9)
box(3.5,2.9,2.6,1.5,"OpenSOS PWA\n(GitHub Pages, HTTPS)\ninstallable - offline",DARK,fs=9)
box(3.5,4.9,2.6,0.7,"Device GPS (Geolocation)",GREEN,fs=9)
box(6.9,2.9,2.9,1.5,"Relay server (Render)\ntoken-gated\nRESEND email API",BLUE,fs=9)
box(6.9,0.7,2.9,0.9,"Trusted contacts\nemail + map link",GREEN,fs=9)
arr(2.7,4.8,3.5,4.0); arr(2.7,3.5,3.5,3.6); arr(2.7,2.2,3.5,3.1)
arr(4.8,4.9,4.8,4.4); arr(6.1,3.65,6.9,3.65); arr(8.35,2.9,8.35,1.6)
ax.text(5,5.7,"OpenSOS system architecture",ha="center",fontsize=13,weight="bold")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig1_architecture.png",dpi=200,bbox_inches="tight"); plt.show()

In [ ]:

# === Figure 2 — app screenshots: core alerting flow ===
montage([("03-home.png","(a) Home / SOS"),("04-countdown.png","(b) Countdown"),
         ("06-alert-success.png","(c) Alert sent"),("11-webhook.png","(d) Wi-Fi setup")],
        "fig2_screens_core.png")

In [ ]:

# === Figure 3 — app screenshots: setup & management ===
montage([("01-welcome.png","(a) Welcome"),("02-onboarding.png","(b) Onboarding"),
         ("08-contacts.png","(c) Contacts"),("12-history.png","(d) History")],
        "fig3_screens_setup.png")

In [ ]:

# === Figure 4 — mobile OS market share ===
fig,ax=plt.subplots(figsize=(7,3.4))
groups=list(OS_SHARE); android=[OS_SHARE[g]["Android"] for g in groups]; ios=[OS_SHARE[g]["iOS"] for g in groups]
y=np.arange(len(groups)); h=0.38
ax.barh(y+h/2,android,h,label="Android",color=GREEN); ax.barh(y-h/2,ios,h,label="iOS",color=DARK)
for i,(a,b) in enumerate(zip(android,ios)):
    ax.text(a+1,i+h/2,f"{a:.1f}%",va="center",fontsize=10); ax.text(b+1,i-h/2,f"{b:.1f}%",va="center",fontsize=10)
ax.set_yticks(y); ax.set_yticklabels(groups); ax.set_xlim(0,80); ax.set_xlabel("Share of mobile devices (%)")
ax.legend(loc="lower right",frameon=False); ax.set_title("Mobile OS market share (StatCounter)",fontsize=12,weight="bold")
for s in ["top","right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig4_os_share.png",dpi=200); plt.show()

In [ ]:

# === Figure 5 — feature availability by platform ===
rows=[r[0] for r in COVERAGE]; M=np.array([r[1] for r in COVERAGE])
cmap={2:GREEN,1:AMBER,0:"#c94b4b"}; lab={2:"Yes",1:"Partial",0:"No"}
fig,ax=plt.subplots(figsize=(8.2,4.2))
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.add_patch(plt.Rectangle((j,M.shape[0]-1-i),0.96,0.92,color=cmap[M[i,j]]))
        ax.text(j+0.48,M.shape[0]-1-i+0.46,lab[M[i,j]],ha="center",va="center",color="white",weight="bold",fontsize=10)
ax.set_xlim(0,3); ax.set_ylim(0,len(rows))
ax.set_xticks(np.arange(3)+0.48); ax.set_xticklabels(COVERAGE_COLS,fontsize=10)
ax.set_yticks(np.arange(len(rows))+0.46); ax.set_yticklabels(rows[::-1],fontsize=10)
ax.xaxis.tick_top(); ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
ax.set_title("OpenSOS feature availability by platform",fontsize=12,weight="bold",pad=28)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig5_coverage.png",dpi=200,bbox_inches="tight"); plt.show()

In [ ]:

# === Figure 6 — cross-platform reach ===
labels=[r[0] for r in REACH]; vals=[r[1] for r in REACH]; colors=[GREY if r[2] else BLUE for r in REACH]
fig,ax=plt.subplots(figsize=(7.4,3.4)); y=np.arange(len(labels))[::-1]
ax.barh(y,vals,color=colors,height=0.6)
for yi,v in zip(y,vals): ax.text(v+0.8,yi,f"{v:.0f}%",va="center",fontsize=10,weight="bold")
ax.set_yticks(y); ax.set_yticklabels(labels,fontsize=10); ax.set_xlim(0,108); ax.set_xlabel("Approx. share of mobile users reached (%)")
ax.set_title("Cross-platform reach of OpenSOS capabilities",fontsize=12,weight="bold")
for s in ["top","right"]: ax.spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/fig6_reach.png",dpi=200); plt.show()

### Done

All six figures are in the `figures/` folder. Re-run any single cell after editing the data block to update just that figure.
